In [2]:
%reload_ext autoreload
%autoreload 2

import warnings, os, sys
warnings.filterwarnings('ignore')
from IPython.display import clear_output

import numpy as np
import pandas as pd
import pickle
import ast
from functools import reduce
from tqdm.auto import tqdm
import networkx as nx

from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import PandasTools
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 600, 400
IPythonConsole.drawOptions.addAtomIndices = False

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default_config = {
    "toImageButtonOptions": {"format": "png", "scale": 5}}


# sys.path.append(os.path.abspath(R"C:\\Users\\Idener\\Sync\\CODE"))
# from aiddtools import fingerprint_tools as fpt
clear_output()

In [2]:
# !curl -L "https://github.com/mibig-secmet/mibig-json/archive/refs/heads/master.zip" -o mibig-json-master.zip
# !unzip mibig-json-master.zip

In [3]:
import json
import glob
import pandas as pd

rows = []

for file in glob.glob("mibig-json-master/data/*.json"):
    with open(file) as f:
        record = json.load(f)

    cluster = record.get("cluster", {})
    compounds = cluster.get("compounds", [])

    compound_names = []
    compound_smiles = []
    database_ids = []
    npa_ids = []

    for compound in compounds:
        compound_names.append(compound.get("compound"))
        compound_smiles.append(compound.get("chem_struct"))

        for database_id in compound.get("database_id", []):
            database_ids.append(database_id)

            if database_id.lower().startswith("npatlas:"):
                npa_ids.append(database_id.split(":", 1)[1])

    rows.append({"mibig_id": cluster.get("mibig_accession"),
        "biosyn_class": ";".join(cluster.get("biosyn_class", [])),
        "organism": cluster.get("organism_name"),
        "ncbi_tax_id": cluster.get("ncbi_tax_id"),
        "status": cluster.get("status"),
        "minimal": cluster.get("minimal"),
        "n_compounds": len(compounds),
        "compound_names": compound_names,
        "compound_smiles": compound_smiles,
        "database_ids": database_ids,
        "npa_ids": sorted(set(npa_ids))})
df_bgc = pd.DataFrame(rows)

df_bgc = df_bgc.explode('compound_smiles').dropna(subset=['compound_smiles'])
PandasTools.AddMoleculeColumnToFrame(df_bgc, 'compound_smiles')
df_bgc = df_bgc.dropna(subset=['ROMol'])
df_bgc['InChIKey'] = df_bgc['ROMol'].apply(Chem.MolToInchiKey)
df_bgc

# # df_bgc.to_csv("mibig_bgc_df.tsv", sep="\t", index=False)
# mibig_npa = df_bgc.explode("npa_ids")
# mibig_npa = mibig_npa[mibig_npa["npa_ids"].notna()]
# mibig_npa = mibig_npa[(mibig_npa["status"] == "active") &
#     (mibig_npa["minimal"] == False)]
# mibig_npa = mibig_npa.rename(columns={"npa_ids": "npa_id"})
# mibig_npa = mibig_npa[["npa_id", "mibig_id", "biosyn_class", "organism", "ncbi_tax_id"]]
# mibig_npa = mibig_npa.drop_duplicates()

# mibig_npa

,mibig_id,biosyn_class,organism,ncbi_tax_id,status,minimal,n_compounds,compound_names,compound_smiles,database_ids,npa_ids,ROMol,InChIKey
2,BGC0001561,NRP,Streptomyces curacoi,146536,active,True,1,[curacomycin],CC[C@H](C)[C@@H]3NC(=O)[C@@H](CC(C)C)NC(=O)[C@...,[],[],<rdkit.Chem.rdchem.Mol object at 0x7efb2ff8eb30>,CUXAULQHGRSWLF-IKVCUCANSA-N
3,BGC0001004,Polyketide,Streptomyces sp. SCSIO 01127,1035052,active,False,1,[lobophorin B],[H][C@@]12C=C[C@]3([H])C(C)=CCC(O[C@H]4C[C@](C...,"[npatlas:NPA011650, pubchem:44566672]",[NPA011650],<rdkit.Chem.rdchem.Mol object at 0x7efb2ff8e9e0>,GRVIKFGPGUQNHQ-OZRRPFOLSA-N
5,BGC0002226,NRP,Aspergillus aculeatus ATCC 16872,690307,active,False,2,"[acu-dioxomorpholine B, acu-dioxomorpholine A]",C=CC(C)(C)[C@]12C[C@@H]3C(=O)O[C@H](CC4=CC=CC=...,[npatlas:NPA032821],[NPA032821],<rdkit.Chem.rdchem.Mol object at 0x7efb2ff8ec80>,HZOLEYOVBYTDAS-LUHVKKMXSA-N
5,BGC0002226,NRP,Aspergillus aculeatus ATCC 16872,690307,active,False,2,"[acu-dioxomorpholine B, acu-dioxomorpholine A]",[H][C@]12NC3=CC=CC=C3[C@@]1(C(C)(C)C=C)CC4C2C(...,[npatlas:NPA032821],[NPA032821],<rdkit.Chem.rdchem.Mol object at 0x7efb2ff8ecf0>,DQGMTSDVSAYSGR-MDZJEUOHSA-N
7,BGC0000693,Saccharide,Bacillus circulans,1397,active,True,2,"[butirosin A, butirosin B]",C1[C@@H]([C@H]([C@@H]([C@H]([C@@H]1NC(=O)[C@H]...,"[pubchem:12302171, pubchem:12302172]",[],<rdkit.Chem.rdchem.Mol object at 0x7efb2ff8ed60>,XEQLFNPSYWZPOW-SVRMBHBBSA-N
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2741,BGC0002160,NRP,Aspergillus flavus,5059,active,True,3,"[aspergillicin A, aspergillicin F, aspergillic...",CC[C@@H](C)[C@@H]1C(=O)N2CCC[C@H]2C(=O)N3CCCC3...,"[npatlas:NPA010186, pubchem:10930650, chemspid...",[NPA010186],<rdkit.Chem.rdchem.Mol object at 0x7efb2ff099e0>,WQAAJWSIZAAGHM-GLSQZRNRSA-N
2741,BGC0002160,NRP,Aspergillus flavus,5059,active,True,3,"[aspergillicin A, aspergillicin F, aspergillic...",CC[C@@H]([C@@H]1C(N2CCC[C@H]2C(N3CCC[C@H]3C(N(...,"[npatlas:NPA010186, pubchem:10930650, chemspid...",[NPA010186],<rdkit.Chem.rdchem.Mol object at 0x7efb2ff09a50>,SLQROOIGZSQKAF-IJHHBHCYSA-N
2742,BGC0001117,NRP,Streptomyces himastatinicus ATCC 53653,457427,active,False,1,[himastatin],[H][C@@]12NC3=CC=C(C=C3[C@]1(O)C[C@@H]1N2C(=O)...,[npatlas:NPA016115],[NPA016115],<rdkit.Chem.rdchem.Mol object at 0x7efb2ff09ac0>,OZPNLJQULOIOGU-UPABGHPRSA-N
2743,BGC0000306,NRP,Streptomyces filamentosus NRRL 11379,457430,active,False,1,[arylomycin],CCCCCCCCCCCCCCCC(=O)N(C)[C@H](CO)C(=O)N[C@H](C...,[pubchem:54586211],[],<rdkit.Chem.rdchem.Mol object at 0x7efb2ff09b30>,HGWCKYGOWQEYSJ-JBWZRHCTSA-N


In [10]:
if not os.path.exists('annotator.py'):
    sys.path.append(os.path.abspath('..'))

df_npa = pd.read_csv('/home/mattia/geass/data/NPAtlas_download_2024_09.tsv', sep='\t')
df_npa = df_npa.rename(columns={
    'npaid': 'NPA_ID',
    'compound_inchikey': 'InChIKey',
    'compound_smiles': 'ISOSMILES',
})
PandasTools.AddMoleculeColumnToFrame(df_npa, 'ISOSMILES')
df_npa = df_npa.dropna(subset=['ROMol'])

df_merge = df_bgc.merge(df_npa, on='InChIKey', how='inner', suffixes=('_bgc', '_npa'))
df_merge = df_merge.drop_duplicates(subset=['InChIKey']).reset_index(drop=True)
df_merge['ROMol'] = df_merge['ROMol_npa']
df_merge = df_merge[['mibig_id', 'NPA_ID', 'InChIKey', 'ISOSMILES', 'ROMol']]
df_merge


,mibig_id,NPA_ID,InChIKey,ISOSMILES,ROMol
0,BGC0001561,NPA028314,CUXAULQHGRSWLF-IKVCUCANSA-N,CC[C@H](C)[C@@H]1NC(=O)[C@@H](CC(C)C)NC(=O)[C@...,<rdkit.Chem.rdchem.Mol object at 0x7efaedd35890>
1,BGC0002226,NPA032821,HZOLEYOVBYTDAS-LUHVKKMXSA-N,C=CC(C)(C)[C@]12C[C@@H]3C(=O)O[C@H](CC4=CC=CC=...,<rdkit.Chem.rdchem.Mol object at 0x7efaedc49970>
2,BGC0002243,NPA033455,CCUJPSDKAUVKIU-UHFFFAOYSA-N,COC1=C(O)C2=C3C(=C1)OC(=O)C1=C3C(=C(O)C=C1C)C(...,<rdkit.Chem.rdchem.Mol object at 0x7efaedb81350>
3,BGC0001145,NPA009917,GMFKDKZZMAPRGJ-GLERVACHSA-N,C/C=C\1/C(=O)NC2(C(=O)N[C@H](C(=O)NCC(=O)N/C(=...,<rdkit.Chem.rdchem.Mol object at 0x7efb2f4c2ac0>
4,BGC0000092,NPA020397,LRWRQTMTYVZKQW-WWDNQWNISA-N,CC[C@@H]1[C@@]([C@@H]([C@H](C(=O)[C@@H](C[C@@]...,<rdkit.Chem.rdchem.Mol object at 0x7efb2f5d8350>
...,...,...,...,...,...
2022,BGC0000521,NPA015244,VIEVUPYQRUJFFA-XFKIZFQISA-N,CCC(C)C1C(=O)NC2CSCC3C(=O)NC(CSCC(C(=O)NC(C(=O...,<rdkit.Chem.rdchem.Mol object at 0x7efb2f58a900>
2023,BGC0000970,NPA000508,FYBGSKAWXCBCIQ-JRAUAZSASA-N,CCCC[C@@H](C)[C@@H]([C@H](C)C(=O)/C(=C/[C@H]([...,<rdkit.Chem.rdchem.Mol object at 0x7efb2f2c3890>
2024,BGC0000959,NPA024793,LIBKJCZUBOETPB-RYUDHWBXSA-N,CC[C@H](C)C[C@H](C)C(=O)C1=C(C(=CNC1=O)C2=CC=C...,<rdkit.Chem.rdchem.Mol object at 0x7efaedd396d0>
2025,BGC0000959,NPA024794,SFWFNVOTIWZXMZ-QWRGUYRKSA-N,CC[C@H](C)C[C@H](C)C(=O)C1=C(C(=CNC1=O)C2=CC(=...,<rdkit.Chem.rdchem.Mol object at 0x7efaedd39660>


In [11]:
from scipy.sparse import csr_matrix, load_npz
from scipy.sparse.csgraph import connected_components

def cluster_from_sparse_similarity(base, df, threshold, method_label):
    ids = np.load(f'{base}_ids.npy', allow_pickle=True)
    sim = load_npz(f'{base}.npz').tocsr().astype(np.float32, copy=False)

    present = df['NPA_ID'].isin(ids)
    df_in = df.loc[present].copy().reset_index(drop=True)
    missing_ids = df.loc[~present, 'NPA_ID'].tolist()
    if missing_ids:
        print(f'Skipping {len(missing_ids)} rows missing from {method_label} ids.')
        print('Examples:', missing_ids[:5])

    order = pd.Index(ids).get_indexer(df_in['NPA_ID'].to_numpy())
    sim = sim[order][:, order].tocsr()
    sim.data = np.clip(sim.data, 0.0, 1.0)

    adj = sim.copy()
    adj.data = (adj.data >= threshold).astype(np.int8)
    adj.setdiag(0)
    adj.eliminate_zeros()

    n_components, labels = connected_components(adj, directed=False, return_labels=True)

    cluster_ids = np.empty(len(labels), dtype=object)
    mapping = {}
    next_id = 1
    for idx, label in enumerate(labels):
        if label not in mapping:
            mapping[label] = str(next_id)
            next_id += 1
        cluster_ids[idx] = mapping[label]

    cluster_col = f'cluster_id_{method_label}_t{int(threshold * 100):02d}'
    out = df_in.copy()
    out[cluster_col] = cluster_ids

    print('Method:', method_label)
    print('Threshold:', threshold)
    print('Clusters:', n_components)
    print('Largest cluster size:', pd.Series(cluster_ids).value_counts().max())
    return out

df_map4 = cluster_from_sparse_similarity('/home/mattia/geass/NPA_analysis/NPA_sim_matrix_map4', df_merge, 0.34, 'map4')
df_map4


Skipping 2 rows missing from map4 ids.
Examples: ['NPA032834', 'NPA033322']
Method: map4
Threshold: 0.34
Clusters: 1029
Largest cluster size: 31


,mibig_id,NPA_ID,InChIKey,ISOSMILES,ROMol,cluster_id_map4_t34
0,BGC0001561,NPA028314,CUXAULQHGRSWLF-IKVCUCANSA-N,CC[C@H](C)[C@@H]1NC(=O)[C@@H](CC(C)C)NC(=O)[C@...,<rdkit.Chem.rdchem.Mol object at 0x7efaedd35890>,1
1,BGC0002226,NPA032821,HZOLEYOVBYTDAS-LUHVKKMXSA-N,C=CC(C)(C)[C@]12C[C@@H]3C(=O)O[C@H](CC4=CC=CC=...,<rdkit.Chem.rdchem.Mol object at 0x7efaedc49970>,2
2,BGC0002243,NPA033455,CCUJPSDKAUVKIU-UHFFFAOYSA-N,COC1=C(O)C2=C3C(=C1)OC(=O)C1=C3C(=C(O)C=C1C)C(...,<rdkit.Chem.rdchem.Mol object at 0x7efaedb81350>,3
3,BGC0001145,NPA009917,GMFKDKZZMAPRGJ-GLERVACHSA-N,C/C=C\1/C(=O)NC2(C(=O)N[C@H](C(=O)NCC(=O)N/C(=...,<rdkit.Chem.rdchem.Mol object at 0x7efb2f4c2ac0>,4
4,BGC0000092,NPA020397,LRWRQTMTYVZKQW-WWDNQWNISA-N,CC[C@@H]1[C@@]([C@@H]([C@H](C(=O)[C@@H](C[C@@]...,<rdkit.Chem.rdchem.Mol object at 0x7efb2f5d8350>,5
...,...,...,...,...,...,...
2020,BGC0000521,NPA015244,VIEVUPYQRUJFFA-XFKIZFQISA-N,CCC(C)C1C(=O)NC2CSCC3C(=O)NC(CSCC(C(=O)NC(C(=O...,<rdkit.Chem.rdchem.Mol object at 0x7efb2f58a900>,1027
2021,BGC0000970,NPA000508,FYBGSKAWXCBCIQ-JRAUAZSASA-N,CCCC[C@@H](C)[C@@H]([C@H](C)C(=O)/C(=C/[C@H]([...,<rdkit.Chem.rdchem.Mol object at 0x7efb2f2c3890>,1028
2022,BGC0000959,NPA024793,LIBKJCZUBOETPB-RYUDHWBXSA-N,CC[C@H](C)C[C@H](C)C(=O)C1=C(C(=CNC1=O)C2=CC=C...,<rdkit.Chem.rdchem.Mol object at 0x7efaedd396d0>,188
2023,BGC0000959,NPA024794,SFWFNVOTIWZXMZ-QWRGUYRKSA-N,CC[C@H](C)C[C@H](C)C(=O)C1=C(C(=CNC1=O)C2=CC(=...,<rdkit.Chem.rdchem.Mol object at 0x7efaedd39660>,188


In [12]:
df_sgo = cluster_from_sparse_similarity('/home/mattia/geass/NPA_analysis/NPA_sim_matrix_sgo', df_merge, 0.70, 'sgo')
df_sgo


Skipping 2 rows missing from sgo ids.
Examples: ['NPA032834', 'NPA033322']
Method: sgo
Threshold: 0.7
Clusters: 882
Largest cluster size: 763


,mibig_id,NPA_ID,InChIKey,ISOSMILES,ROMol,cluster_id_sgo_t70
0,BGC0001561,NPA028314,CUXAULQHGRSWLF-IKVCUCANSA-N,CC[C@H](C)[C@@H]1NC(=O)[C@@H](CC(C)C)NC(=O)[C@...,<rdkit.Chem.rdchem.Mol object at 0x7efaedd35890>,1
1,BGC0002226,NPA032821,HZOLEYOVBYTDAS-LUHVKKMXSA-N,C=CC(C)(C)[C@]12C[C@@H]3C(=O)O[C@H](CC4=CC=CC=...,<rdkit.Chem.rdchem.Mol object at 0x7efaedc49970>,2
2,BGC0002243,NPA033455,CCUJPSDKAUVKIU-UHFFFAOYSA-N,COC1=C(O)C2=C3C(=C1)OC(=O)C1=C3C(=C(O)C=C1C)C(...,<rdkit.Chem.rdchem.Mol object at 0x7efaedb81350>,3
3,BGC0001145,NPA009917,GMFKDKZZMAPRGJ-GLERVACHSA-N,C/C=C\1/C(=O)NC2(C(=O)N[C@H](C(=O)NCC(=O)N/C(=...,<rdkit.Chem.rdchem.Mol object at 0x7efb2f4c2ac0>,4
4,BGC0000092,NPA020397,LRWRQTMTYVZKQW-WWDNQWNISA-N,CC[C@@H]1[C@@]([C@@H]([C@H](C(=O)[C@@H](C[C@@]...,<rdkit.Chem.rdchem.Mol object at 0x7efb2f5d8350>,1
...,...,...,...,...,...,...
2020,BGC0000521,NPA015244,VIEVUPYQRUJFFA-XFKIZFQISA-N,CCC(C)C1C(=O)NC2CSCC3C(=O)NC(CSCC(C(=O)NC(C(=O...,<rdkit.Chem.rdchem.Mol object at 0x7efb2f58a900>,1
2021,BGC0000970,NPA000508,FYBGSKAWXCBCIQ-JRAUAZSASA-N,CCCC[C@@H](C)[C@@H]([C@H](C)C(=O)/C(=C/[C@H]([...,<rdkit.Chem.rdchem.Mol object at 0x7efb2f2c3890>,881
2022,BGC0000959,NPA024793,LIBKJCZUBOETPB-RYUDHWBXSA-N,CC[C@H](C)C[C@H](C)C(=O)C1=C(C(=CNC1=O)C2=CC=C...,<rdkit.Chem.rdchem.Mol object at 0x7efaedd396d0>,882
2023,BGC0000959,NPA024794,SFWFNVOTIWZXMZ-QWRGUYRKSA-N,CC[C@H](C)C[C@H](C)C(=O)C1=C(C(=CNC1=O)C2=CC(=...,<rdkit.Chem.rdchem.Mol object at 0x7efaedd39660>,882


In [13]:
from scipy.stats import chi2_contingency
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, v_measure_score

map4_col = 'cluster_id_map4_t34'
sgo_col = 'cluster_id_sgo_t70'
df_eval = df_map4[['mibig_id', 'NPA_ID', map4_col]].merge(
    df_sgo[['NPA_ID', sgo_col]],
    on='NPA_ID',
    how='inner',
)

def score_against_bgc(df, cluster_col):
    labels_true = df['mibig_id'].astype(str)
    labels_pred = df[cluster_col].astype(str)
    contingency = pd.crosstab(labels_true, labels_pred)
    chi2 = chi2_contingency(contingency, correction=False)[0]
    n = contingency.to_numpy().sum()
    r, k = contingency.shape
    den = n * max(1, min(r - 1, k - 1))
    cramers_v = np.sqrt(chi2 / den) if den > 0 else np.nan
    return {
        'ARI': adjusted_rand_score(labels_true, labels_pred),
        'NMI': normalized_mutual_info_score(labels_true, labels_pred),
        'V_measure': v_measure_score(labels_true, labels_pred),
        'Cramers_V': cramers_v,
        'n_rows': len(df),
        'n_bgc': labels_true.nunique(),
        'n_clusters': labels_pred.nunique(),
    }

summary = pd.DataFrame([
    {'method': 'MAP4', **score_against_bgc(df_eval, map4_col)},
    {'method': 'SGO', **score_against_bgc(df_eval, sgo_col)},
]).set_index('method')

print('Shared rows:', len(df_eval))
summary


Shared rows: 2025


,ARI,NMI,V_measure,Cramers_V,n_rows,n_bgc,n_clusters
method,,,,,,,
MAP4,0.604653,0.964428,0.964428,0.953456,2025,1192,1029
SGO,0.008173,0.784542,0.784542,0.885568,2025,1192,882


In [14]:
df_cluster_sizes = pd.DataFrame({
    'MAP4': df_map4['cluster_id_map4_t34'].value_counts(),
    'SGO': df_sgo['cluster_id_sgo_t70'].value_counts(),
}).fillna(0).astype(int)

display(df_cluster_sizes.sort_values(['MAP4', 'SGO'], ascending=False).head(20))


,MAP4,SGO
157,31,2
200,20,2
102,19,1
203,19,1
49,19,1
23,17,1
701,15,1
132,13,1
175,13,1
138,12,1


In [16]:
from scipy.sparse import load_npz
from scipy.sparse.csgraph import connected_components
from scipy.stats import chi2_contingency
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, v_measure_score

def sweep_thresholds(base, df, method_label, thresholds):
    ids = np.load(f'{base}_ids.npy', allow_pickle=True)
    sim = load_npz(f'{base}.npz').tocsr().astype(np.float32, copy=False)
    present = df['NPA_ID'].isin(ids)
    df_in = df.loc[present].copy().reset_index(drop=True)
    order = pd.Index(ids).get_indexer(df_in['NPA_ID'].to_numpy())
    sim = sim[order][:, order].tocsr()
    sim.data = np.clip(sim.data, 0.0, 1.0)

    rows = []
    labels_true = df_in['mibig_id'].astype(str)
    for threshold in thresholds:
        adj = sim.copy()
        adj.data = (adj.data >= threshold).astype(np.int8)
        adj.setdiag(0)
        adj.eliminate_zeros()
        _, labels = connected_components(adj, directed=False, return_labels=True)
        labels_pred = pd.Series(labels).astype(str)
        contingency = pd.crosstab(labels_true, labels_pred)
        chi2 = chi2_contingency(contingency, correction=False)[0]
        n = contingency.to_numpy().sum()
        r, k = contingency.shape
        den = n * max(1, min(r - 1, k - 1))
        cramers_v = np.sqrt(chi2 / den) if den > 0 else np.nan

        rows.append({
            'method': method_label,
            'threshold': threshold,
            'ARI': adjusted_rand_score(labels_true, labels_pred),
            'NMI': normalized_mutual_info_score(labels_true, labels_pred),
            'V_measure': v_measure_score(labels_true, labels_pred),
            'Cramers_V': cramers_v,
            'n_clusters': int(pd.Series(labels).nunique()),
            'largest_cluster': int(pd.Series(labels).value_counts().max()),
        })
    return pd.DataFrame(rows)

map4_thresholds = np.round(np.linspace(0.20, 0.90, 15), 2)
sgo_thresholds = np.round(np.linspace(0.20, 0.90, 15), 2)

df_sweep = pd.concat([
    sweep_thresholds('/home/mattia/geass/NPA_analysis/NPA_sim_matrix_map4', df_merge, 'MAP4', map4_thresholds),
    sweep_thresholds('/home/mattia/geass/NPA_analysis/NPA_sim_matrix_sgo', df_merge, 'SGO', sgo_thresholds),
], ignore_index=True)

df_overview = df_sweep.sort_values(['method', 'threshold']).reset_index(drop=True)
print(df_overview.to_string(index=False))


method  threshold      ARI      NMI  V_measure  Cramers_V  n_clusters  largest_cluster
  MAP4       0.20 0.047569 0.877680   0.877680   0.972702         701              383
  MAP4       0.25 0.135386 0.926657   0.926657   0.964985         849              207
  MAP4       0.30 0.546143 0.958524   0.958524   0.957616         964               41
  MAP4       0.35 0.627817 0.965939   0.965939   0.949451        1050               31
  MAP4       0.40 0.637698 0.967874   0.967874   0.941405        1107               30
  MAP4       0.45 0.666004 0.969066   0.969066   0.930159        1169               23
  MAP4       0.50 0.672607 0.970224   0.970224   0.936304        1247               19
  MAP4       0.55 0.639104 0.969182   0.969182   0.947064        1317               19
  MAP4       0.60 0.624649 0.968156   0.968156   0.958440        1395               15
  MAP4       0.65 0.569362 0.966026   0.966026   0.967008        1481               15
  MAP4       0.70 0.484169 0.962636   0.962